# Faster R-CNN fine-tuning
        **vision › detection** · notebook `fasterrcnn_finetune`

        **Kapan dipakai:** Kelas atau domain yang TIDAK ada di COCO dan ada sedikit data berkotak: fine-tune dari bobot COCO.

        **Jebakan umum:** Bila kelasnya sudah ada di COCO (mis. orang), fine-tuning hampir tidak menambah skor dibanding model bawaan. Anotasi kotak harus konsisten; lr terlalu besar merusak bobot pretrained.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Penn-Fudan (170 gambar pejalan kaki, kotak + mask); 120 latih / 50 uji tetap. Metrik: **AP50** (kotak benar bila IoU ≥ 0.5). Bobot dibaca dari `pretrained/` (offline).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
MODEL_KEY = "fasterrcnn_finetune"
OUT = out_dir("vision", "detection", MODEL_KEY)
SEED = 42; set_seed(SEED)
SCORE_THR = 0.05                 # ambang skor deteksi (rendah untuk AP; naikkan untuk hasil bersih)
EPOCHS = 5

## 2. Data: gambar + kotak pejalan kaki (dari mask instance)
120 gambar pertama latih, 50 terakhir uji (tetap).

In [ ]:
from superdb.vision_det import load_pennfudan, split_idx, ap50, box_iou
D = load_pennfudan(); itr, ite = split_idx(len(D), 50)
imgs = [d["img"] for d in D]; boxes = [d["boxes"] for d in D]
print(len(D), "gambar | rata-rata orang per gambar:", round(float(np.mean([len(b) for b in boxes])), 2))
import matplotlib.patches as mp
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, i in zip(ax, (0, 1, 2)):
    a.imshow(imgs[i]); [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="lime", lw=2)) for b in boxes[i]]; a.axis("off")
plt.tight_layout(); plt.show()

## 3. Model Faster R-CNN MobileNetV3 (bobot COCO lokal)

In [ ]:
from pathlib import Path
import torch
from superdb.vision_det import fasterrcnn, detect_fasterrcnn, train_fasterrcnn
from superdb.vision import device
print("device:", device())

## 4. Fine-tuning pada 120 gambar latih
Kepala diganti: 2 kelas (latar + orang). Pembanding: model COCO tanpa fine-tune.

In [ ]:
base = fasterrcnn(pretrained_coco=True).to(device())
ap_base = ap50(detect_fasterrcnn(base, [imgs[i] for i in ite], SCORE_THR, label=1), [boxes[i] for i in ite])["AP50"]
model = fasterrcnn(pretrained_coco=True, num_classes=2)
hist = train_fasterrcnn(model, [imgs[i] for i in itr], [boxes[i] for i in itr], epochs=EPOCHS, lr=5e-3, batch=4, seed=SEED)
preds = detect_fasterrcnn(model, [imgs[i] for i in ite], SCORE_THR, label=1)
metrics = ap50(preds, [boxes[i] for i in ite]); metrics["AP50_tanpa_finetune(COCO)"] = ap_base; display(show_table(metrics))

## 5. Lihat hasil (hijau = benar, merah = prediksi)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, k in zip(ax, (0, 1, 2)):
    i = ite[k]; a.imshow(imgs[i]); a.axis("off")
    [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="lime", lw=2)) for b in boxes[i]]
    b_, s_ = preds[k]; [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="red", lw=1.5, ls="--")) for b, s in zip(b_, s_) if s > 0.5]
plt.tight_layout(); plt.savefig(OUT / "detections.png", dpi=130); plt.show()
save_json({"metrics": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

In [ ]:
torch.save(model.state_dict(), OUT / "model.pt"); m2 = fasterrcnn(pretrained_coco=False, num_classes=2); m2.load_state_dict(torch.load(OUT / "model.pt", map_location="cpu")); m2.to(device())
a = detect_fasterrcnn(m2, [imgs[ite[0]]], SCORE_THR, label=1)[0][0]; b = detect_fasterrcnn(model, [imgs[ite[0]]], SCORE_THR, label=1)[0][0]
assert np.allclose(a, b, atol=1e-2), "model muat-ulang beda"; print("model tersimpan dan terverifikasi")

> **Baca hasilnya dengan jujur:** kelas 'person' sudah ada di COCO, jadi model bawaan sudah hampir sempurna dan fine-tuning hampir tidak menambah skor (bandingkan `AP50` dengan `AP50_tanpa_finetune`). Fine-tuning baru berarti untuk **kelas/domain baru** (mis. jenis produk, objek satelit) yang tidak ada di COCO.

## Langkah berikutnya
- Skor ambang rendah menghasilkan banyak kotak palsu; pilih ambang dari kurva precision-recall pada validation.
- Data lomba: ubah loader (`load_pennfudan`) menjadi pembaca anotasi lomba (COCO JSON/Pascal VOC/YOLO). Evaluasi `ap50` bisa dipakai apa adanya.
- Gambar besar dengan objek kecil: naikkan `imgsz`.